# 03 Modifying the machine: element settings, groups, RF, errors

This notebook runs the five scripts of `03_modifying/` in order with `%run`, so it runs exactly the same code as the command line (`python 03_modifying/NN_name.py`). Each script's docstring says what it shows, what it needs and what it checks.

15, 17, 18 and the solenoid in 19 use small lattices built in Python (with `02_tracking/simple_lattice.py`). 16 and the dogleg in 19 run JFEL's bunch compressor as a line of its own, from a `.def` written into `output/` by `chicane.py`. Every check compares a tracked beam with a formula worked out from first principles.

Everything here is tier A (pip only); 18 adds an elegant check when the simcodes image is there. The whole notebook takes under a minute.

## 15 Read and change element settings

`framework["QD0"]` is the LAURA element. `getElement`/`modifyElement` read and set one parameter, by short name (`"k1l"`) or full path (`"magnetic.k1l"`); `getElementType`/`setElementType` do every element of a type; `apply_changes` takes a dictionary. `save_changes_file` writes what changed (old and new values) and `load_changes_file` applies it to a fresh lattice.

`getElementType` lists elements in load order, not lattice order, and `setElementType` pairs its values with that order, so build the values from `getElementType`.

The check flips every quadrupole: the new strengths must appear in the Ocelot deck, and the horizontal beam must now follow the original lattice's vertical matrix.

In [ ]:
%run 15_modify_elements.py

## 16 A chicane as a group

A `.def`'s `groups:` block ties elements together. A `chicane` group has `.angle`, an `.r56` estimate and `set_angle(a)`, which re-lays out the dipoles and everything between them. The script tracks a chirp-free beam through JFEL's chicane at 0.13 and 0.10 rad, measures $R_{56}$ by fitting each particle's change in $z$ against its starting $x$, $x'$ and $\delta$, and compares with the exact formula

$$R_{56} = 2\left[2\rho(\tan\theta - \theta) + D\,\frac{\tan^2\theta}{\cos\theta}\right] + \frac{S}{\gamma^2}$$

where the last term is the velocity spread over the line's length $S$. The group's `.r56` is the small-angle formula and reads 1.1% low. `set_angle` keeps each dipole's entrance and exit along the axis and lets the arc follow, so at the angle the chicane already has it changes nothing (it once lengthened the dipoles by 0.28% there: ISSUES.md #27).

In [ ]:
%run 16_groups_chicane.py

## 17 Scan an RF cavity's phase

One S-band travelling-wave cavity, scanned in phase with `modifyElement`. The mean energy gain must follow $\Delta E = V\cos(\phi - \phi_0)$ with the crest $\phi_0 = 0$ (SIMBA's convention, whatever the code's) and $V = A\,(N + 3.8)\,\ell_\mathrm{cell}/\sqrt{2}$ for a travelling-wave cavity of field amplitude $A$ (a standing-wave cavity's `field_amplitude` is its voltage). The scan avoids $\pm 90°$, where Cheetah's (and at $+90°$ Ocelot's) cavity gives NaN.

Fit the energy, not the momentum: at 100 MeV they differ by $1.3\times10^{-5}$ in the gain.

In [ ]:
%run 17_rf_phase_scan.py

In [ ]:
from IPython.display import Image, display
display(Image(filename="../output/17_rf_phase_scan/phase_scan.png"))

## 18 Misaligned quadrupoles and random errors

`physical.error.position.x` (and `.y`, `.z`, `rotation.*`) moves an element off axis, in every code. A quad displaced by $d$ acts on $x - d$, so the centroid map is $u \to M_q(u - d) + d$. The script checks that for one quad in four codes, then for seeded random offsets on every quad of three FODO cells.

SIMBA's own `loadElementErrors`/`setSeedValue` work only in elegant and only on strengths; on main they raise `KeyError` (`XFAIL #25`, printed only when the simcodes image is there).

In [ ]:
%run 18_errors_and_misalignments.py

## 19 Dispersion from a dogleg, rotation in a solenoid

The first half of JFEL's chicane is a dogleg. Its rectangular dipoles return every particle parallel to the axis, so at the middle screen $\eta' = 0$ exactly and

$$\eta = 2\rho\left(\frac{1}{\cos\theta} - 1\right) + D\,\frac{\tan\theta}{\cos^2\theta}.$$

A solenoid rotates a beam by the Larmor angle $S_{0L}/(2B\rho)$; a flat, nearly parallel beam shows it as a change of tilt.

In [ ]:
%run 19_dispersion_and_solenoid.py